# **Prédire le contenu d'un panier**

## Import

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.multiclass import OneVsRestClassifier

## Modèle

In [ ]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

top_500_products = df["product_name"].value_counts().head(500).index
df_filtered = df[df["product_name"].isin(top_500_products)]

grouped_by_orders = df_filtered.groupby("order_id").agg({
    "product_name": list,
    "user_id": "first",
    "order_number": "first",
    "order_dow": "first",
    "order_hour_of_day": "first",
    "days_since_prior_order": "first",
    
}).reset_index()

grouped_by_orders["size_of_the_basket"] = grouped_by_orders["product_name"].map(len)
print(grouped_by_orders)

feature_cols = [
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "size_of_the_basket"
]

X = pd.get_dummies(grouped_by_orders[feature_cols])

y = grouped_by_orders["product_name"]

mlb = MultiLabelBinarizer(sparse_output=True)
y_matrice = mlb.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_matrice, test_size=0.2, random_state=42
)

model = OneVsRestClassifier(
    BernoulliNB()
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(y_pred)